# Qwen3.6-27B Paper-Grade SAE — L11 / L31 / L55

**Target**: n=65536 (13× expansion), k=128, AuxK dead-feature mitigation (Gao et al. 2024), 200M tokens per layer, streaming 70/20/10 FineWeb-Edu + OpenThoughts + OpenMath mix, incremental HF checkpoints every 10M tokens so a Colab crash loses ≤10 min of compute.

**Artifacts**: `caiovicentino1/qwen36-27b-sae-papergrade` — sae_lens-format safetensors, Neuronpedia-ingestion-ready.

**References**: Gao et al. 2024 ([arxiv:2406.04093](https://arxiv.org/abs/2406.04093)), Gemma Scope ([arxiv:2408.05147](https://arxiv.org/abs/2408.05147)), SAELens format ([jbloomAus/SAELens](https://github.com/jbloomAus/SAELens)).

**Hardware**: RTX PRO 6000 Blackwell 96 GB. Expected wall-time: ~6-8h per SAE at 200M tokens; 3 SAEs share one forward pass so total ≈ 8-12h end-to-end.

## 0. Install deps

In [ ]:
import sys, subprocess, shutil, os
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

# Qwen3.6 ships with model_type='qwen3_5' — not yet in stable transformers.
# Install from main if missing, then tell the user to restart runtime.
try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    has_qwen35 = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception:
    has_qwen35 = False

if not has_qwen35:
    print('Installing transformers from source for qwen3_5 support...')
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'tqdm', 'sentencepiece', 'tokenizers', 'protobuf', 'hf_transfer')
    pip('uninstall', '-y', '-q', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','-q','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','-q','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]
    print('\n*** RESTART RUNTIME NOW, then re-run this cell. ***')
else:
    print(f'transformers {transformers.__version__} has qwen3_5 ✓')

# HF auth (Colab secret first, then env var fallback)
try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        os.environ['HF_TOKEN'] = t
except Exception:
    t = os.environ.get('HF_TOKEN')
if t:
    from huggingface_hub import login
    login(token=t, add_to_git_credential=False)
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    print('HF auth OK')
else:
    print('⚠ No HF_TOKEN found — set it in Colab secrets or env before checkpoint cell')

## 1. Config — all tunable knobs in one place

Override any of these before running subsequent cells.

In [ ]:
import os, torch

# ---- Model ----
MODEL_ID      = 'Qwen/Qwen3.6-27B'
LAYERS        = [11, 31, 55]              # residual stream read points
D_MODEL       = 5120

# ---- SAE architecture (paper-grade) ----
N_FEATURES    = 65_536                    # 13x expansion — Gemma-Scope-27B parity
K_TOPK        = 128                       # Gao et al. interpretability sweet spot
K_AUX         = 2_560                     # d_model/2 — AuxK (Gao et al. §2.4)
ALPHA_AUX     = 1.0 / 32                  # AuxK loss coefficient
DEAD_TOKENS   = 10_000_000                # dead feature threshold

# ---- Training ----
TOKEN_BUDGET  = 200_000_000               # per SAE (user said '100M+' — 200M = 2x safety)
BUFFER_TOKENS = 2_000_000                 # activation buffer in RAM (~20 GB @ bf16)
BATCH_SIZE    = 4096                      # SAE training batch (tokens)
LR_PEAK       = 2e-4
LR_FLOOR      = 6e-5
WARMUP_STEPS  = 5_000
GRAD_CLIP     = 1.0

# ---- Activation extraction (forward pass through 27B) ----
SEQ_LEN       = 1024                      # tokens per sequence
FWD_BATCH     = 2                         # sequences per forward pass (2×1024 = 2048 tok)

# ---- Corpus mix ----
CORPUS_MIX = [
    ('HuggingFaceFW/fineweb-edu',             'sample-10BT', 0.70),
    ('open-thoughts/OpenThoughts-114k',       'default',     0.20),
    ('nvidia/OpenMathInstruct-2',             'default',     0.10),
]

# ---- Checkpointing ----
CKPT_EVERY_TOK = 10_000_000               # every 10M tokens → HF upload
HF_REPO        = 'caiovicentino1/qwen36-27b-sae-papergrade'
LOCAL_CKPT_DIR = '/content/sae_ckpt'
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

# ---- Misc ----
DEVICE        = 'cuda'
DTYPE_MODEL   = torch.bfloat16            # 27B model
DTYPE_SAE     = torch.float32             # SAE params (Gao et al. recommend fp32)
SEED          = 42

torch.manual_seed(SEED)
print(f'Config: n={N_FEATURES}, k={K_TOPK}, tokens_per_sae={TOKEN_BUDGET/1e6:.0f}M, layers={LAYERS}')

## 2. Load Qwen3.6-27B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=DTYPE_MODEL,
    device_map=DEVICE,
    attn_implementation='sdpa',
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print(f'VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 3. Discover + verify layer paths

Qwen3.6-27B (loaded via `AutoModelForImageTextToText`) nests decoder layers under `model.language_model.layers.N` even though it's text-only. Never hardcode.

In [ ]:
def get_layer_module(m, idx):
    for path in [('model','language_model','layers'),
                 ('language_model','layers'),
                 ('model','layers')]:
        try:
            cur = m
            for p in path:
                cur = getattr(cur, p)
            return cur[idx]
        except AttributeError:
            continue
    raise RuntimeError('Could not locate decoder layers')

for L in LAYERS:
    mod = get_layer_module(model, L)
    print(f'L{L}: {type(mod).__name__}')

# Sanity: confirm hook fires and records nonzero residuals
_probe = {}
def _mk_probe(L):
    def _h(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        _probe[L] = h.detach().float().norm().item()
    return _h
handles = [get_layer_module(model, L).register_forward_hook(_mk_probe(L)) for L in LAYERS]
ids = tok('The quick brown fox', return_tensors='pt').input_ids.to(DEVICE)
with torch.no_grad():
    model(ids)
for h in handles: h.remove()
print('Probe norms:', _probe)
assert all(v > 0 for v in _probe.values()), 'Hooks did not fire!'

## 4. Corpus streaming — 70/20/10 mix

Streams tokens from three HF datasets with weighted interleaving. FineWeb-Edu (raw text) + OpenThoughts (chat template with thinking) + OpenMath (math CoT).

In [ ]:
from datasets import load_dataset
import random

random.seed(SEED)

def _load_stream(repo, split):
    try:
        return load_dataset(repo, split, split='train', streaming=True)
    except Exception:
        return load_dataset(repo, split='train', streaming=True)

def _text_fineweb(row):
    return row.get('text', '')

def _text_openthoughts(row):
    conv = row.get('conversations') or row.get('messages') or []
    if not conv:
        return row.get('text', '')
    msgs = [{'role': m.get('from', m.get('role', 'user')).replace('human','user').replace('gpt','assistant'),
             'content': m.get('value', m.get('content', ''))} for m in conv]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, enable_thinking=True)
    except Exception:
        return '\n\n'.join(m['content'] for m in msgs)

def _text_openmath(row):
    q = row.get('problem') or row.get('question') or ''
    a = row.get('generated_solution') or row.get('solution') or row.get('answer') or ''
    return f'Problem: {q}\n\nSolution: {a}'

CORPUS_EXTRACTORS = {
    'HuggingFaceFW/fineweb-edu':       _text_fineweb,
    'open-thoughts/OpenThoughts-114k': _text_openthoughts,
    'nvidia/OpenMathInstruct-2':       _text_openmath,
}

def mixed_text_stream():
    streams = []
    weights = []
    for repo, split, w in CORPUS_MIX:
        it = iter(_load_stream(repo, split))
        streams.append((repo, it))
        weights.append(w)
    while True:
        idx = random.choices(range(len(streams)), weights=weights, k=1)[0]
        repo, it = streams[idx]
        try:
            row = next(it)
        except StopIteration:
            streams[idx] = (repo, iter(_load_stream(repo, CORPUS_MIX[idx][1])))
            row = next(streams[idx][1])
        txt = CORPUS_EXTRACTORS[repo](row)
        if txt and len(txt) > 50:
            yield txt

# Smoke test
_g = mixed_text_stream()
for _ in range(3):
    s = next(_g)
    print(f'[{len(s)} chars] {s[:100]!r}...')

## 5. Activation streamer — one forward pass, 3 layers captured

Runs Qwen forward on packed sequences of `SEQ_LEN` tokens, captures residuals at all 3 target layers simultaneously. Yields `(activations_L11, activations_L31, activations_L55)` each of shape `(FWD_BATCH * SEQ_LEN, D_MODEL)`.

In [ ]:
class LayerTap:
    def __init__(self, model, layers):
        self.buf = {L: None for L in layers}
        self.handles = []
        for L in layers:
            mod = get_layer_module(model, L)
            self.handles.append(mod.register_forward_hook(self._mk_hook(L)))
    def _mk_hook(self, L):
        def _h(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            self.buf[L] = h.detach().to(torch.bfloat16)
        return _h
    def close(self):
        for h in self.handles:
            h.remove()

def pack_sequences(text_gen, n_seq, seq_len):
    """Pack tokens from text stream into n_seq sequences of exactly seq_len tokens."""
    out = []
    carry = []
    while len(out) < n_seq:
        if len(carry) < seq_len:
            carry.extend(tok(next(text_gen), add_special_tokens=False).input_ids)
            continue
        out.append(carry[:seq_len])
        carry = carry[seq_len:]
    return torch.tensor(out, dtype=torch.long, device=DEVICE)

def activation_stream(text_gen, token_budget):
    """Yields dicts {L: tensor(B*T, D)} each batch. Stops after token_budget tokens emitted."""
    tap = LayerTap(model, LAYERS)
    emitted = 0
    try:
        while emitted < token_budget:
            ids = pack_sequences(text_gen, FWD_BATCH, SEQ_LEN)
            with torch.no_grad():
                model(ids)
            chunk = {L: tap.buf[L].reshape(-1, D_MODEL) for L in LAYERS}
            emitted += chunk[LAYERS[0]].shape[0]
            yield chunk, emitted
    finally:
        tap.close()

# Smoke test: one batch
_g = mixed_text_stream()
_s = activation_stream(_g, token_budget=FWD_BATCH * SEQ_LEN)
_chunk, _em = next(_s)
for L in LAYERS:
    t = _chunk[L]
    print(f'L{L}: shape={tuple(t.shape)}, mean_norm={t.float().norm(dim=-1).mean():.3f}, std={t.float().std():.3f}')

## 6. TopK SAE with AuxK dead-feature loss

Gao et al. 2024 recipe:
- Hard TopK activation (no L1 penalty)
- AuxK loss: dead features attempt to reconstruct main-pass residual (dead = didn't fire in last 10M tokens)
- Decoder column norm reprojected to 1.0 every step
- b_dec initialized to geometric median of first 16k activations

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k, k_aux):
        super().__init__()
        self.d_in, self.n, self.k, self.k_aux = d_in, n, k, k_aux
        # Tied init: W_dec rows are unit-normed, W_enc = W_dec.T (Gao et al. App B)
        W = torch.randn(n, d_in) / (d_in ** 0.5)
        W = W / W.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        self.W_dec = nn.Parameter(W)                                    # [n, d_in]
        self.W_enc = nn.Parameter(W.T.clone().contiguous())             # [d_in, n]
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

    def encode_pre(self, x):
        return (x - self.b_dec) @ self.W_enc + self.b_enc              # [B, n]

    def forward(self, x, dead_mask=None):
        pre = self.encode_pre(x)
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        x_hat = z @ self.W_dec + self.b_dec
        main_loss = (x - x_hat).pow(2).mean()
        # AuxK: dead features reconstruct residual
        aux_loss = x.new_zeros(())
        if dead_mask is not None and dead_mask.sum() >= self.k_aux:
            residual = (x - x_hat).detach()
            pre_dead = pre.masked_fill(~dead_mask.unsqueeze(0), float('-inf'))
            v_aux, i_aux = pre_dead.topk(self.k_aux, dim=-1)
            z_aux = torch.zeros_like(pre)
            z_aux.scatter_(-1, i_aux, F.relu(v_aux))
            res_hat = z_aux @ self.W_dec                                # no b_dec shift
            aux_loss = (residual - res_hat).pow(2).mean()
        return x_hat, z, top_i, main_loss, aux_loss

    @torch.no_grad()
    def renorm_decoder(self):
        self.W_dec.data /= self.W_dec.data.norm(dim=-1, keepdim=True).clamp_min(1e-8)

    @torch.no_grad()
    def set_b_dec_geomedian(self, samples, n_iter=40):
        """Weiszfeld geometric median — robust b_dec init for heavy-tailed residuals."""
        med = samples.mean(0)
        for _ in range(n_iter):
            d = (samples - med).norm(dim=-1).clamp_min(1e-8)
            w = 1.0 / d
            med = (samples * w.unsqueeze(1)).sum(0) / w.sum()
        self.b_dec.data = med.to(self.b_dec.dtype)

# Smoke test
_sae = TopKSAE(D_MODEL, 4096, 32, 256).to(DEVICE, DTYPE_SAE)
_x = torch.randn(64, D_MODEL, device=DEVICE, dtype=DTYPE_SAE)
_dm = torch.zeros(4096, dtype=torch.bool, device=DEVICE); _dm[:500] = True
_xh, _z, _ti, _ml, _al = _sae(_x, _dm)
print(f'smoke: x_hat={tuple(_xh.shape)}, main={_ml:.4f}, aux={_al:.4f}, L0={(_z>0).sum(-1).float().mean():.0f}')
del _sae, _x, _dm, _xh, _z, _ti, _ml, _al

## 7. HF checkpoint utilities — crash-safe

`save_ckpt` writes a local `.pt` + uploads to HF repo as `sae_L{L}_resume.safetensors` (weights+optimizer) and `sae_L{L}_latest.safetensors` (weights only).

`load_ckpt` attempts HF → local → fresh init.

Crash recovery: re-run this notebook from Cell 1, subsequent cells will detect the existing checkpoint and resume.

In [ ]:
from huggingface_hub import HfApi, login, hf_hub_download
from safetensors.torch import save_file, load_file
import json, shutil

# Auth (set HF_TOKEN env or run `huggingface-cli login` first)
if os.environ.get('HF_TOKEN'):
    login(os.environ['HF_TOKEN'])
hfapi = HfApi()

try:
    hfapi.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False)
    print(f'Repo ready: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'create_repo warning: {e}')

def _ckpt_paths(L):
    return dict(
        weights = f'sae_L{L}_latest.safetensors',
        resume  = f'sae_L{L}_resume.pt',
        cfg     = f'sae_L{L}_cfg.json',
    )

def save_ckpt(L, sae, optim, scheduler, step, tokens_seen, last_fired):
    """Save weights (safetensors, sae_lens keys) + resume state (pt)."""
    p = _ckpt_paths(L)

    # 1. sae_lens-format weights
    weights_path = f'{LOCAL_CKPT_DIR}/{p["weights"]}'
    save_file({
        'W_enc': sae.W_enc.detach().cpu().contiguous(),
        'W_dec': sae.W_dec.detach().cpu().contiguous(),
        'b_enc': sae.b_enc.detach().cpu().contiguous(),
        'b_dec': sae.b_dec.detach().cpu().contiguous(),
    }, weights_path)

    # 2. cfg.json (sae_lens / Neuronpedia compatible)
    cfg_path = f'{LOCAL_CKPT_DIR}/{p["cfg"]}'
    with open(cfg_path, 'w') as f:
        json.dump({
            'architecture': 'topk',
            'd_in': D_MODEL,
            'd_sae': N_FEATURES,
            'k': K_TOPK,
            'k_aux': K_AUX,
            'alpha_aux': ALPHA_AUX,
            'hook_name': f'model.language_model.layers.{L}',
            'model_name': MODEL_ID,
            'activation_fn_str': 'relu',
            'tokens_seen': tokens_seen,
            'step': step,
        }, f, indent=2)

    # 3. Optimizer + scheduler + dead tracker (for resume)
    resume_path = f'{LOCAL_CKPT_DIR}/{p["resume"]}'
    torch.save({
        'sae_state': sae.state_dict(),
        'optim': optim.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler is not None else None,
        'step': step,
        'tokens_seen': tokens_seen,
        'last_fired': last_fired.cpu(),
    }, resume_path)

    # 4. Upload all three
    for local_name in [p['weights'], p['cfg'], p['resume']]:
        try:
            hfapi.upload_file(
                path_or_fileobj=f'{LOCAL_CKPT_DIR}/{local_name}',
                path_in_repo=local_name,
                repo_id=HF_REPO,
                commit_message=f'L{L} ckpt step={step} tokens={tokens_seen/1e6:.1f}M',
            )
        except Exception as e:
            print(f'  upload fail {local_name}: {e}')
    print(f'  ✓ saved L{L} @ step={step} tokens={tokens_seen/1e6:.1f}M')

def load_ckpt(L, sae, optim, scheduler):
    """Try HF → return (step, tokens_seen, last_fired) or (0, 0, None) if none."""
    p = _ckpt_paths(L)
    try:
        local = hf_hub_download(HF_REPO, p['resume'], local_dir=LOCAL_CKPT_DIR)
        state = torch.load(local, map_location='cpu', weights_only=False)
        sae.load_state_dict(state['sae_state'])
        optim.load_state_dict(state['optim'])
        if scheduler is not None and state.get('scheduler'):
            scheduler.load_state_dict(state['scheduler'])
        print(f'  ✓ resumed L{L} from step={state["step"]} tokens={state["tokens_seen"]/1e6:.1f}M')
        return state['step'], state['tokens_seen'], state['last_fired'].to(DEVICE)
    except Exception as e:
        print(f'  (no ckpt for L{L}, fresh init: {type(e).__name__})')
        return 0, 0, None

print('Checkpoint utilities ready.')

## 8. Init SAEs + optimizers + schedulers + geomedian b_dec

Collect one buffer of activations, fit b_dec via geometric median (16k samples), then init Adam + cosine-with-warmup scheduler.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR
import math

total_steps = TOKEN_BUDGET // BATCH_SIZE
print(f'Planned steps per SAE: {total_steps:,} ({TOKEN_BUDGET/1e6:.0f}M tokens)')

saes, optims, scheds, last_fired_map = {}, {}, {}, {}
for L in LAYERS:
    sae = TopKSAE(D_MODEL, N_FEATURES, K_TOPK, K_AUX).to(DEVICE, DTYPE_SAE)
    optim = torch.optim.Adam(sae.parameters(), lr=LR_PEAK, betas=(0.9, 0.999), eps=1e-8)
    def lr_lambda(step, warm=WARMUP_STEPS, total=total_steps, floor=LR_FLOOR/LR_PEAK):
        if step < warm:
            return step / max(1, warm)
        prog = (step - warm) / max(1, total - warm)
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    sched = LambdaLR(optim, lr_lambda)
    saes[L], optims[L], scheds[L] = sae, optim, sched

# Try resuming from HF
resume_steps = {}
resume_tokens = {}
for L in LAYERS:
    s, t, lf = load_ckpt(L, saes[L], optims[L], scheds[L])
    resume_steps[L] = s
    resume_tokens[L] = t
    last_fired_map[L] = lf if lf is not None else torch.full((N_FEATURES,), -DEAD_TOKENS, device=DEVICE, dtype=torch.long)

# If starting fresh on any layer, init b_dec via geomedian
fresh_layers = [L for L in LAYERS if resume_steps[L] == 0]
if fresh_layers:
    print(f'Geomedian init for fresh layers: {fresh_layers}')
    _g = mixed_text_stream()
    _samples = {L: [] for L in fresh_layers}
    _collected = 0
    for chunk, emitted in activation_stream(_g, token_budget=16_384):
        for L in fresh_layers:
            _samples[L].append(chunk[L])
        _collected = emitted
        if _collected >= 16_384:
            break
    for L in fresh_layers:
        S = torch.cat(_samples[L])[:16_384].to(DEVICE, DTYPE_SAE)
        saes[L].set_b_dec_geomedian(S)
        print(f'  L{L} b_dec norm={saes[L].b_dec.norm():.3f}')
    del _samples
    torch.cuda.empty_cache()

## 9. Training loop — 3 SAEs, shared forward pass, HF checkpoints every 10M tokens

One Qwen forward pass → capture activations at L11/L31/L55 → each SAE takes its slice and does a training step. Training runs until each layer reaches `TOKEN_BUDGET`.

**Checkpoint cadence**: every `CKPT_EVERY_TOK` tokens we upload weights + optimizer state to HF. If Colab crashes, re-running the notebook resumes from last checkpoint.

In [ ]:
from tqdm.auto import tqdm
import time

# Start from the max of all layers' resume points (they may diverge if a crash hit mid-step)
start_tokens = min(resume_tokens.values()) if resume_tokens else 0
print(f'Resume from {start_tokens/1e6:.1f}M tokens → target {TOKEN_BUDGET/1e6:.0f}M per layer')

text_gen = mixed_text_stream()
remaining = TOKEN_BUDGET - start_tokens

# Shared counters
global_tokens = start_tokens
last_ckpt_tokens = start_tokens
log_every_steps = 100
t0 = time.time()

# Running log
metrics = {L: {'step': 0, 'main': 0.0, 'aux': 0.0, 'var_expl': 0.0, 'dead': 0} for L in LAYERS}

pbar = tqdm(total=remaining, unit='tok', unit_scale=True, smoothing=0.1, desc='SAE train')

for chunk, emitted_cum in activation_stream(text_gen, token_budget=remaining):
    batch_tokens = chunk[LAYERS[0]].shape[0]
    global_tokens += batch_tokens

    for L in LAYERS:
        sae, opt, sch = saes[L], optims[L], scheds[L]
        x = chunk[L].to(DEVICE, DTYPE_SAE, non_blocking=True)

        # Dead mask: features whose last-fired token is older than DEAD_TOKENS
        dead_mask = (last_fired_map[L] < (global_tokens - DEAD_TOKENS))

        # Shuffle + split into mini-batches of BATCH_SIZE
        perm = torch.randperm(x.shape[0], device=DEVICE)
        x = x[perm]
        for i in range(0, x.shape[0], BATCH_SIZE):
            xb = x[i:i+BATCH_SIZE]
            if xb.shape[0] < 64:
                continue
            xh, z, top_i, main_l, aux_l = sae(xb, dead_mask)
            loss = main_l + ALPHA_AUX * aux_l
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sae.parameters(), GRAD_CLIP)
            opt.step()
            sch.step()
            sae.renorm_decoder()

            # Update last-fired tracker
            fired = torch.unique(top_i)
            last_fired_map[L][fired] = global_tokens

            # Metrics
            with torch.no_grad():
                var = 1.0 - (xb - xh).pow(2).sum() / (xb - xb.mean(0)).pow(2).sum().clamp_min(1e-8)
            m = metrics[L]
            m['step'] += 1
            m['main'] = 0.98 * m['main'] + 0.02 * main_l.item() if m['step'] > 1 else main_l.item()
            m['aux']  = 0.98 * m['aux']  + 0.02 * (aux_l.item() if torch.is_tensor(aux_l) else aux_l) if m['step'] > 1 else (aux_l.item() if torch.is_tensor(aux_l) else 0.0)
            m['var_expl'] = 0.98 * m['var_expl'] + 0.02 * var.item() if m['step'] > 1 else var.item()
            m['dead'] = int(dead_mask.sum().item())

    pbar.update(batch_tokens)

    # Log
    if metrics[LAYERS[0]]['step'] % log_every_steps == 0:
        msg = ' | '.join([f'L{L}: ve={metrics[L]["var_expl"]:.3f} dead={metrics[L]["dead"]}' for L in LAYERS])
        pbar.set_postfix_str(msg)

    # Checkpoint
    if global_tokens - last_ckpt_tokens >= CKPT_EVERY_TOK:
        pbar.write(f'=== Checkpoint @ {global_tokens/1e6:.1f}M tokens ===')
        for L in LAYERS:
            save_ckpt(L, saes[L], optims[L], scheds[L],
                      step=metrics[L]['step'],
                      tokens_seen=global_tokens,
                      last_fired=last_fired_map[L])
        last_ckpt_tokens = global_tokens

    if global_tokens >= TOKEN_BUDGET:
        break

pbar.close()

# Final checkpoint
print('=== Final checkpoint ===')
for L in LAYERS:
    save_ckpt(L, saes[L], optims[L], scheds[L],
              step=metrics[L]['step'],
              tokens_seen=global_tokens,
              last_fired=last_fired_map[L])

elapsed = time.time() - t0
print(f'Total training time: {elapsed/3600:.2f}h ({global_tokens/1e6:.0f}M tokens × {len(LAYERS)} SAEs)')

## 10. Held-out validation — var_expl + L0 + dead fraction

Run on 1M fresh tokens (different seed) to confirm no overfit.

In [ ]:
random.seed(SEED + 1)
val_gen = mixed_text_stream()

val_stats = {L: {'mse': 0.0, 'var_total': 0.0, 'residual_sq': 0.0, 'l0_sum': 0, 'l0_n': 0, 'fired': set()} for L in LAYERS}
VAL_TOKENS = 1_000_000

for chunk, emitted in activation_stream(val_gen, token_budget=VAL_TOKENS):
    for L in LAYERS:
        sae = saes[L]
        x = chunk[L].to(DEVICE, DTYPE_SAE)
        with torch.no_grad():
            xh, z, top_i, _, _ = sae(x)
            s = val_stats[L]
            s['residual_sq'] += (x - xh).pow(2).sum().item()
            s['var_total']   += (x - x.mean(0)).pow(2).sum().item()
            s['l0_sum']      += (z > 0).sum().item()
            s['l0_n']        += x.shape[0]
            s['fired'].update(top_i.unique().cpu().tolist())
    if emitted >= VAL_TOKENS:
        break

print(f'\nHeld-out validation ({VAL_TOKENS/1e6:.1f}M tokens):')
print(f'{"Layer":<6} {"var_expl":>10} {"L0":>8} {"alive":>8} {"dead%":>8}')
val_report = {}
for L in LAYERS:
    s = val_stats[L]
    var_expl = 1.0 - s['residual_sq'] / s['var_total']
    l0 = s['l0_sum'] / s['l0_n']
    alive = len(s['fired'])
    dead_pct = 100.0 * (1 - alive / N_FEATURES)
    val_report[L] = {'var_expl': var_expl, 'l0': l0, 'alive': alive, 'dead_pct': dead_pct}
    print(f'L{L:<5} {var_expl:>10.4f} {l0:>8.1f} {alive:>8d} {dead_pct:>7.2f}%')

# Save validation report
with open(f'{LOCAL_CKPT_DIR}/val_report.json', 'w') as f:
    json.dump({str(L): v for L, v in val_report.items()}, f, indent=2)
hfapi.upload_file(
    path_or_fileobj=f'{LOCAL_CKPT_DIR}/val_report.json',
    path_in_repo='val_report.json',
    repo_id=HF_REPO,
    commit_message=f'Held-out validation ({VAL_TOKENS/1e6:.0f}M tokens)',
)

## 11. Cleanup — remove bloat, keep only sae_lens-format weights + cfg

Resume files (optimizer state) are ~2-3GB each. Delete from repo after final upload.

In [ ]:
from huggingface_hub import list_repo_files
try:
    files = list_repo_files(HF_REPO)
    for f in files:
        if f.endswith('_resume.pt'):
            try:
                hfapi.delete_file(f, repo_id=HF_REPO, commit_message='cleanup resume state after convergence')
                print(f'  deleted {f}')
            except Exception as e:
                print(f'  delete fail {f}: {e}')
except Exception as e:
    print(f'list warning: {e}')

print('\nFinal repo layout:')
for f in sorted(list_repo_files(HF_REPO)):
    print(f'  {f}')

print(f'\n✓ Paper-grade SAEs ready: https://huggingface.co/{HF_REPO}')